Data Prep for Vehicle Emission Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler, LabelEncoder
from sklearn.preprocessing import OneHotEncoder

sns.set_theme(style="whitegrid")
pd.set_option('display.max_columns', None)

## Step 1: Initial Data Exploration

In [ ]:
data = pd.read_csv('data/raw/1-vehicle_emission_dataset.csv')
print("Shape:", data.shape)
data.head()

In [ ]:
print("--- Data Types ---")
print(data.dtypes)
print("\n--- Basic Statistics ---")
data.describe()

Handling Missing Values

In [ ]:
# Identify missing values per column
print("Missing values per column:")
print(data.isna().sum())
print(f"\nTotal missing: {data.isna().sum().sum()}")

In [ ]:
# Drop columns where more than 50% of values are missing
threshold = len(data) * 0.5
data_clean = data.dropna(thresh=threshold, axis=1)

# For remaining numeric columns, impute with median
numeric_cols = data_clean.select_dtypes(include=np.number).columns
data_clean[numeric_cols] = data_clean[numeric_cols].fillna(data_clean[numeric_cols].median())

# For categorical columns, impute with mode
categorical_cols = data_clean.select_dtypes(include='object').columns
for col in categorical_cols:
    data_clean[col] = data_clean[col].fillna(data_clean[col].mode()[0])

print("Missing values after imputation:")
print(data_clean.isna().sum())
print(f"\nShape after handling missing values: {data_clean.shape}")

## Step 3: Dealing with Duplicates

In [ ]:
print(f"Duplicate rows found: {data_clean.duplicated().sum()}")
data_clean = data_clean.drop_duplicates()
print(f"Shape after removing duplicates: {data_clean.shape}")

## Step 4: Handling Categorical Data

In [ ]:
# Standardize text: strip whitespace and lowercase for consistency
for col in data_clean.select_dtypes(include='object').columns:
    data_clean[col] = data_clean[col].str.strip().str.lower()

# --- Ordinal Encoding ---
# 'Emission Level' has a natural order: low < medium < high
emission_order = {'low': 0, 'medium': 1, 'high': 2}
data_clean['Emission Level'] = data_clean['Emission Level'].map(emission_order)
print("Emission Level mapping applied:")
print(data_clean['Emission Level'].value_counts().sort_index())

# --- Nominal Encoding (One-Hot) ---
# Nominal features: Vehicle Type, Fuel Type, Road Type, Traffic Conditions
nominal_cols = ['Vehicle Type', 'Fuel Type', 'Road Type', 'Traffic Conditions']
data_encoded = pd.get_dummies(data_clean, columns=nominal_cols, drop_first=False)
print(f"\nShape after one-hot encoding: {data_encoded.shape}")
data_encoded.head()

## Step 5: Data Manipulation & Outlier Management

In [ ]:
# --- Discretization: bin Age of Vehicle into groups ---
data_encoded['Age Group'] = pd.cut(
    data_encoded['Age of Vehicle'],
    bins=[0, 5, 10, 15, 20, 30],
    labels=['0-5 yrs', '6-10 yrs', '11-15 yrs', '16-20 yrs', '20+ yrs']
)
print("Age Group distribution:")
print(data_encoded['Age Group'].value_counts().sort_index())

# --- Outlier Capping (±3 standard deviations) ---
numeric_features = data_encoded.select_dtypes(include=np.number).columns.tolist()
# Exclude the target and binary dummy columns
exclude_cols = ['Emission Level'] + [c for c in numeric_features if data_encoded[c].nunique() == 2]
cap_cols = [c for c in numeric_features if c not in exclude_cols]

outlier_counts = {}
for col in cap_cols:
    mean, std = data_encoded[col].mean(), data_encoded[col].std()
    lower, upper = mean - 3 * std, mean + 3 * std
    n_outliers = ((data_encoded[col] < lower) | (data_encoded[col] > upper)).sum()
    if n_outliers > 0:
        outlier_counts[col] = n_outliers
    data_encoded[col] = data_encoded[col].clip(lower, upper)

print(f"\nOutliers capped per column: {outlier_counts}")
print(f"Shape after outlier treatment: {data_encoded.shape}")

## Step 6: Data Partitioning (Train/Test Split)

In [ ]:
# Drop non-numeric/derived columns before splitting
drop_cols = ['Age Group']
df_model = data_encoded.drop(columns=drop_cols)

X = df_model.drop(columns=['Emission Level'])
y = df_model['Emission Level']

# Stage 1: hold out 20% as the test set
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Stage 2: split remainder into 75% train / 25% validation → 60/20/20 overall
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp
)

print(f"Training set:    {X_train.shape[0]} rows ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Validation set:  {X_val.shape[0]} rows ({X_val.shape[0]/len(X)*100:.1f}%)")
print(f"Test set:        {X_test.shape[0]} rows ({X_test.shape[0]/len(X)*100:.1f}%)")
print(f"\nTarget class distribution (train):\n{y_train.value_counts().sort_index()}")
print(f"\nTarget class distribution (val):\n{y_val.value_counts().sort_index()}")
print(f"\nTarget class distribution (test):\n{y_test.value_counts().sort_index()}")

## Step 7: Feature Scaling

In [ ]:
# Scale only continuous numeric features (exclude binary dummy columns)
scale_cols = [c for c in X_train.select_dtypes(include=np.number).columns
              if X_train[c].nunique() > 2]

# --- Standardization (preferred: mean=0, std=1) ---
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_val_scaled   = X_val.copy()
X_test_scaled  = X_test.copy()

# Fit ONLY on training data, then transform all sets to prevent data leakage
X_train_scaled[scale_cols] = scaler.fit_transform(X_train[scale_cols])
X_val_scaled[scale_cols]   = scaler.transform(X_val[scale_cols])
X_test_scaled[scale_cols]  = scaler.transform(X_test[scale_cols])

print("Standardization applied to:", scale_cols)
print(f"\nTrain feature means (should be ~0):\n{X_train_scaled[scale_cols].mean().round(4)}")
print(f"\nTrain feature std (should be ~1):\n{X_train_scaled[scale_cols].std().round(4)}")

# --- MinMax Normalization (for reference) ---
minmax = MinMaxScaler()
X_train_norm = X_train.copy()
X_val_norm   = X_val.copy()
X_test_norm  = X_test.copy()
X_train_norm[scale_cols] = minmax.fit_transform(X_train[scale_cols])
X_val_norm[scale_cols]   = minmax.transform(X_val[scale_cols])
X_test_norm[scale_cols]  = minmax.transform(X_test[scale_cols])
print(f"\nMinMax range check (train) — min:\n{X_train_norm[scale_cols].min().round(4)}")
print(f"MinMax range check (train) — max:\n{X_train_norm[scale_cols].max().round(4)}")

## Step 8: Visual Exploration

In [ ]:
# --- Univariate: Distributions of key numeric features ---
cont_features = ['CO2 Emissions', 'Speed', 'Mileage', 'Engine Size',
                 'Temperature', 'Humidity', 'Age of Vehicle']

fig, axes = plt.subplots(3, 3, figsize=(15, 10))
axes = axes.flatten()

for i, col in enumerate(cont_features):
    sns.histplot(data_clean[col], kde=True, ax=axes[i], color='steelblue')
    axes[i].set_title(f'Distribution of {col}')
    axes[i].set_xlabel(col)

# Hide unused subplot
for j in range(len(cont_features), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Univariate Distributions of Key Numeric Features', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# --- Multivariate: CO2 Emissions by Emission Level and Fuel Type ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar plot: Average CO2 by Fuel Type
fuel_co2 = data_clean.groupby('Fuel Type')['CO2 Emissions'].mean().sort_values(ascending=False)
fuel_co2.plot.bar(ax=axes[0], color='coral', edgecolor='black')
axes[0].set_title('Average CO2 Emissions by Fuel Type')
axes[0].set_xlabel('Fuel Type')
axes[0].set_ylabel('Avg CO2 Emissions')
axes[0].tick_params(axis='x', rotation=0)

# Bar plot: Emission Level class counts
emission_labels = {0: 'Low', 1: 'Medium', 2: 'High'}
y_counts = y.map(emission_labels).value_counts()
y_counts[['Low', 'Medium', 'High']].plot.bar(ax=axes[1], color='mediumseagreen', edgecolor='black')
axes[1].set_title('Emission Level Class Distribution')
axes[1].set_xlabel('Emission Level')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
# --- Correlation Heatmap of numeric features ---
numeric_cols_plot = ['Engine Size', 'Age of Vehicle', 'Mileage', 'Speed',
                     'Acceleration', 'Temperature', 'Humidity', 'Wind Speed',
                     'Air Pressure', 'CO2 Emissions', 'NOx Emissions',
                     'PM2.5 Emissions', 'VOC Emissions', 'SO2 Emissions', 'Emission Level']

corr = data_clean[numeric_cols_plot].corr()

plt.figure(figsize=(14, 10))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Correlation Heatmap of Numeric Features', fontsize=14)
plt.tight_layout()
plt.show()

## Export: Processed Dataset

### Use Case → Feature Mapping

| Use Case | Key Features | Target |
|---|---|---|
| **Compare Driving Behaviors** | `Vehicle Type`, `Fuel Type`, `Road Type`, `Traffic Conditions`, `Speed`, `Mileage`, `Acceleration` | `Emission Level` |
| **Identify Performance Bottlenecks** | `CO2 Emissions`, `NOx Emissions`, `PM2.5 Emissions`, `VOC Emissions`, `SO2 Emissions`, `Engine Size`, `Age of Vehicle` | `Emission Level` |

The full encoded dataset is saved for flexible use across both use cases. Scaled train/test splits are saved separately for direct model consumption.

In [ ]:
import os
os.makedirs('data/processed', exist_ok=True)

# --- Full processed dataset (encoded, outliers capped, target as ordinal int) ---
# Excludes derived/discretized 'Age Group'; includes all model-ready columns
full_processed = data_encoded.drop(columns=['Age Group'], errors='ignore')
full_processed.to_csv('data/processed/1-vehicle_emission_processed.csv', index=False)
print(f"Saved full processed dataset: {full_processed.shape}")
print(f"Columns: {full_processed.columns.tolist()}")

In [ ]:
# --- Scaled train/validation/test splits (StandardScaler, leakage-free) ---
X_train_scaled.to_csv('data/processed/1-vehicle_emission_X_train.csv', index=False)
X_val_scaled.to_csv('data/processed/1-vehicle_emission_X_val.csv', index=False)
X_test_scaled.to_csv('data/processed/1-vehicle_emission_X_test.csv', index=False)
y_train.to_csv('data/processed/1-vehicle_emission_y_train.csv', index=False, header=True)
y_val.to_csv('data/processed/1-vehicle_emission_y_val.csv', index=False, header=True)
y_test.to_csv('data/processed/1-vehicle_emission_y_test.csv', index=False, header=True)

print("Saved train/validation/test splits:")
print(f"  X_train: {X_train_scaled.shape}  →  1-vehicle_emission_X_train.csv")
print(f"  X_val:   {X_val_scaled.shape}   →  1-vehicle_emission_X_val.csv")
print(f"  X_test:  {X_test_scaled.shape}   →  1-vehicle_emission_X_test.csv")
print(f"  y_train / y_val / y_test saved.")
print("\nTarget encoding: 0=Low, 1=Medium, 2=High")